In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import nltk
from nltk.corpus import stopwords
from fakenews.config import load_config, resolve

nltk.download("stopwords", quiet=True)
STOP = set(stopwords.words("english"))

sns.set_style("darkgrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 13

cfg = load_config()
df = pd.read_csv(resolve(cfg["paths"]["clean_data"]))
FIG = resolve(cfg["paths"]["reports_dir"]) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
df.shape, df["label"].value_counts()

In [ ]:
for lbl in (0, 1):
    print(f"\n=========== LABEL {lbl} ===========")
    for t in df[df.label == lbl].sample(4, random_state=1)["content"]:
        print("-", t[:300].replace("\n", " "), "\n")

In [ ]:
ax = sns.countplot(x="label", data=df)
ax.set_title("Class distribution (after cleaning)")
plt.savefig(FIG / "class_balance.png", bbox_inches="tight")

In [ ]:
def avg_word(text):
    words = text.split()
    return sum(len(w) for w in words) / len(words) if words else 0

df["word_count"] = df["content"].str.split().str.len()
df["stop_count"] = df["content"].apply(lambda t: sum(w in STOP for w in t.lower().split()))
df["avg_word_len"] = df["content"].apply(avg_word)

df[["word_count", "stop_count", "avg_word_len"]].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, col, rng in zip(axes, ["word_count", "stop_count", "avg_word_len"],
                        [(0, 2000), (0, 800), (3, 8)]):
    sns.histplot(data=df, x=col, hue="label", binrange=rng, multiple="dodge", ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(FIG / "text_stats.png", bbox_inches="tight")

In [ ]:
df.groupby("label")[["word_count", "stop_count", "avg_word_len"]].median()

In [ ]:
from collections import Counter
import re

def top_words(texts, n=15):
    c = Counter()
    for t in texts:
        c.update(w for w in re.findall(r"[a-z]+", t.lower()) if w not in STOP and len(w) > 2)
    return c.most_common(n)

for lbl in (0, 1):
    print(f"Label {lbl}:", top_words(df[df.label == lbl]["content"].sample(5000, random_state=0)))

In [ ]:
checks = {
    "Reuters": r"\(Reuters\)|Reuters",
    "21st Century Wire / Share": r"21st Century Wire|Share this",
    "Twitter handles/pics": r"pic\.twitter\.com",
    "Image caption": r"Featured image",
}
rows = []
for name, pat in checks.items():
    hit = df["content"].str.contains(pat, case=False, regex=True)
    rows.append({"pattern": name, **{f"label_{l}_%": round(hit[df.label == l].mean() * 100, 1) for l in (0, 1)}})
pd.DataFrame(rows)

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, lbl in zip(axes, (0, 1)):
    text = " ".join(df[df.label == lbl]["content"].sample(3000, random_state=0))
    wc = WordCloud(width=800, height=450, stopwords=STOP, max_words=100,
                   background_color="white", random_state=10).generate(text)
    ax.imshow(wc); ax.axis("off"); ax.set_title(f"Label {lbl}")
plt.savefig(FIG / "wordclouds.png", bbox_inches="tight")